# 实验 7 — dlt 元数据：pipeline 存了什么？

**目标**：深入 dlt 在 DuckDB 里写入的内部表，理解 `_dlt_load_id`、`_dlt_id`、`_dlt_loads` 各代表什么。

In [ ]:
import duckdb

DB = "../data/sandbox.duckdb"
con = duckdb.connect(DB)

print("=== raw_ecb schema 的所有表 ===")
con.execute("SHOW TABLES FROM raw_ecb").df()

In [ ]:
print("=== _dlt_loads：每次 pipeline.run() 的记录 ===")
con.execute("""
    SELECT load_id, schema_name, status, inserted_at, schema_version_hash
    FROM raw_ecb._dlt_loads
    ORDER BY inserted_at
""").df()

In [ ]:
print("=== _dlt_version：schema 版本历史 ===")
con.execute("""
    SELECT version, engine_version, inserted_at, version_hash
    FROM raw_ecb._dlt_version
    ORDER BY inserted_at
""").df()

In [ ]:
print("=== _dlt_load_id 关联：哪批数据来自哪次 load ===")
con.execute("""
    SELECT
        dr._dlt_load_id,
        l.inserted_at AS load_time,
        COUNT(*) AS rows_in_batch
    FROM raw_ecb.daily_rates dr
    JOIN raw_ecb._dlt_loads l ON dr._dlt_load_id = l.load_id
    GROUP BY 1, 2
    ORDER BY load_time
""").df()

In [ ]:
print("=== _dlt_id：每行的唯一标识 ===")
con.execute("""
    SELECT date, currency, rate, _dlt_id, _dlt_load_id
    FROM raw_ecb.daily_rates
    ORDER BY date DESC
    LIMIT 5
""").df()

## 思考

- `_dlt_load_id` 是 load 级别 ID（整个 `pipeline.run()` 共享一个）
- `_dlt_id` 是行级别唯一 ID（base64 随机串）
- 通过 `_dlt_load_id` join `_dlt_loads` 可以追溯任意一行数据是什么时候、哪次 pipeline 写入的
- `stg_ecb_rates.sql` 把 `_dlt_load_id` 透传为 `loaded_at_load_id`，这就是 dlt metadata 进入 dbt 模型的方式

In [ ]:
import subprocess

result = subprocess.run(
    ["uv", "run", "dlt", "pipeline", "ecb_rates_pipeline", "info"],
    capture_output=True, text=True, cwd=".."
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)